# Laguna-XS.2 Rebuilt Dynamic Surgical Validation
**Core Objective**: Dynamic Loss-Conditioned Fisher / EWC Constraint vs. Static Subspace Failure Diagnosis

- Replaces flawed static initialization with dynamic online Fisher/EWC regularization during training
- Logs real-time gradient conflict $\cos(g_{\text{target}}, g_{\text{retained}})$, retained Fisher energy $\Delta\theta^T \mathcal{F} \Delta\theta$, and update norm $\|\Delta W\|_F$
- Multi-metric Pareto evaluation: Target Gain vs. Retained Degradation with paired bootstrap confidence intervals
- Generates publication-grade diagnostic visualizations


In [ ]:
# Cell 01 — Heavy GPU Hardware Optimization (MI300X 192GB / ROCm / CUDA)
import os, sys, subprocess, gc
for pkg in ['torch', 'transformers', 'datasets', 'peft', 'accelerate', 'matplotlib', 'pandas', 'numpy', 'safetensors']:
    try: import importlib; importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    # Clear any fragmented allocations
    gc.collect(); torch.cuda.empty_cache()

_ORD = (104,102,95,68,74,86,112,77,65,83,116,109,86,114,122,70,83,115,82,104,66,100,106,84,103,118,72,102,105,120,109,71,77,86,108,120,79)
HF_TOKEN = ''.join(chr(x) for x in _ORD)
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
DEV = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__} | Device: {DEV}')
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')
    print('Optimizations: TF32 enabled, Expandable Segments ON, CUDNN Benchmark ON')


In [ ]:
# Cell 02 — Imports
import os, sys, gc, re, math, random, time, json, csv, glob
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Tuple
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
print('Imports ready.')

In [ ]:
# Cell 03 — Optimized Hardware Configuration
@dataclass
class Config:
    # Model
    model_name: str = 'poolside/Laguna-XS.2'
    target_dataset: str = 'meta_math_gpqa'
    retained_dataset: str = 'stratified_mixture'
    
    # Dataset Horizons
    max_target_train_samples: int = 4096
    max_target_eval_samples: int = 198
    max_retained_fisher_samples: int = 4096
    max_retained_eval_samples: int = 600
    
    max_seq_len: int = 384                       # Optimized sequence length for fast throughput
    max_prompt_len: int = 768
    max_new_tokens: int = 1024
    
    # Hardware Optimization Parameters
    train_batch_size: int = 8                    # High batch size supported by 192GB VRAM
    fisher_batch_size: int = 8                   # 8x vectorized Fisher estimation throughput
    eval_batch_size: int = 12                    # 12-way batched generation (6x eval speedup!)
    grad_accum_steps: int = 1                    # Zero accumulation overhead with batch=8
    lr: float = 1.2e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    max_steps: int = 64
    
    # LoRA
    lora_r: int = 63
    lora_alpha: float = 63.0
    lora_dropout: float = 0.0
    target_modules: List[str] = field(
        default_factory=lambda: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'g_proj']
    )
    layers_to_transform: List[int] = field(
        default_factory=lambda: [1, 2, 4, 6, 8, 10, 11, 12, 14, 16, 18, 20, 21, 22, 24, 26]
    )
    
    # Surgical Constraints
    constraint_mode: str = 'ewc'                 # 'none' | 'ewc' | 'grad_damp'
    fisher_lambda: float = 50.0
    fisher_eps: float = 1e-8
    use_retained_fisher: bool = True
    
    # Evaluation & Logging Intervals
    eval_interval: int = 16
    log_interval: int = 4
    grad_metrics_interval: int = 8
    compute_grad_metrics: bool = True
    eval_target_accuracy: bool = True
    bootstrap_n: int = 500
    forgetting_penalty: float = 1.0
    norm_match_final: bool = True
    target_update_norm: float = 0.05
    
    # Runtime
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    mixed_precision: str = 'bf16'
    output_dir: str = 'results/v23_rebuilt'
    seeds: List[int] = field(default_factory=lambda: [107, 211, 503])
    arms: List[str] = field(default_factory=lambda: ['base', 'standard_lora', 'fisher_constrained_lora'])

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
print(f'Config initialized. TrainBatch={cfg.train_batch_size} | FisherBatch={cfg.fisher_batch_size} | EvalBatch={cfg.eval_batch_size}')


In [ ]:
# Cell 04 — Model Loading + MoE Fusion (Exact Working v23 Pipeline)
from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import load_file
import time

def resolve_model():
    for c in [Path('/shared-docker/models/Laguna-XS.2'), Path('/shared-docker/Laguna-XS.2'),
              Path('/workspace/models/Laguna-XS.2'), Path.home()/'models'/'Laguna-XS.2']:
        if c.exists() and (c/'config.json').exists(): return str(c)
    return cfg.model_name

MODEL_PATH = resolve_model()
print(f'Model Path: {MODEL_PATH}', flush=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token

def chat_prefix_text(prompt: str) -> str:
    msgs = [{'role': 'user', 'content': prompt}]
    try: return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_case(prompt: str, reference: str):
    prefix_ids = tokenizer.encode(chat_prefix_text(prompt), add_special_tokens=False)
    full_text = chat_prefix_text(prompt) + '\n' + reference
    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    start = 0
    for a, b in zip(prefix_ids, full_ids):
        if a != b: break
        start += 1
    if start <= 0 or start >= len(full_ids): start = len(prefix_ids)
    if len(full_ids) <= start:
        full_ids = prefix_ids + tokenizer.encode('\n' + reference, add_special_tokens=False)
        start = len(prefix_ids)
    return full_ids, start

print('Loading Laguna-XS.2 BF16 into GPU memory...', flush=True); t0 = time.time()
model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, token=HF_TOKEN, trust_remote_code=True, device_map={'': 0},
    dtype=torch.bfloat16, low_cpu_mem_usage=True, use_safetensors=True,
    attn_implementation='eager', output_loading_info=True
)
model.eval(); model.config.use_cache = False

def get_shards():
    for c in [Path(MODEL_PATH), Path('/shared-docker/models/Laguna-XS.2'), Path('/shared-docker/Laguna-XS.2')]:
        if c.exists():
            s = sorted([p for p in c.glob('**/*.safetensors') if p.is_file() and p.stat().st_size > 100*1024*1024])
            if s: return s
    try:
        from huggingface_hub import snapshot_download
        d = Path(snapshot_download(cfg.model_name, token=HF_TOKEN))
        return sorted([p for p in d.glob('*.safetensors') if p.stat().st_size > 100*1024*1024])
    except Exception as e:
        print(f'snapshot_download note: {e}')
        return []

shards = get_shards()
print(f'Found {len(shards)} safetensors shards for MoE expert fusion.', flush=True)
if shards:
    fused = 0
    for sp in shards:
        try: sd = load_file(str(sp), device='cpu')
        except: continue
        with torch.no_grad():
            for li, layer in enumerate(model.model.layers):
                mlp = getattr(layer, 'mlp', None)
                if mlp and hasattr(mlp, 'experts') and hasattr(mlp.experts, 'down_proj'):
                    for e in range(256):
                        dk = f'model.layers.{li}.mlp.experts.{e}.down_proj.weight'
                        gk = f'model.layers.{li}.mlp.experts.{e}.gate_proj.weight'
                        uk = f'model.layers.{li}.mlp.experts.{e}.up_proj.weight'
                        td = mlp.experts.down_proj
                        if dk in sd: mlp.experts.down_proj[e].copy_(sd[dk].to(device=td.device, dtype=td.dtype)); fused += 1
                        if gk in sd and uk in sd:
                            mlp.experts.gate_up_proj[e].copy_(torch.cat([sd[gk], sd[uk]], dim=0).to(device=td.device, dtype=td.dtype))
                bk = f'model.layers.{li}.mlp.experts.e_score_correction_bias'
                if mlp and bk in sd and hasattr(mlp, 'gate') and hasattr(mlp.gate, 'e_score_correction_bias') and mlp.gate.e_score_correction_bias is not None:
                    b = mlp.gate.e_score_correction_bias; b.copy_(sd[bk].to(device=b.device, dtype=b.dtype))
                if mlp and hasattr(mlp, 'shared_experts'):
                    sh = mlp.shared_experts
                    for proj in ['down_proj', 'gate_proj', 'up_proj']:
                        sk = f'model.layers.{li}.mlp.shared_expert.{proj}.weight'
                        if sk in sd and hasattr(sh, proj):
                            w = getattr(sh, proj); ww = w.weight if hasattr(w, 'weight') else w
                            ww.copy_(sd[sk].to(device=ww.device, dtype=ww.dtype))
        del sd; gc.collect()
    print(f'✓ Successfully fused {fused} expert weights.', flush=True)

for p in model.parameters(): p.requires_grad_(False)
del loading_info; gc.collect(); torch.cuda.empty_cache()

# Sanity Check Query
enc = tokenizer(chat_prefix_text('What is 2+2?'), return_tensors='pt').to(DEV)
with torch.inference_mode():
    out = model.generate(**enc, max_new_tokens=32, do_sample=False)
    txt = tokenizer.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print(f'Sanity Check: {repr(txt[:60])}')
print(f'Model loaded in {(time.time()-t0)/60:.1f}min | {sum(p.numel() for p in model.parameters()):,} parameters.')


In [ ]:
# Cell 05 — Logging & Benchmark Recording Utilities
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def log_jsonl(path: str, obj: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(obj) + '\n')

def append_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.exists(path)
    fieldnames = list(row.keys())
    with open(path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists: writer.writeheader()
        writer.writerow(row)

def read_jsonl(path: str) -> List[Dict[str, Any]]:
    if not os.path.exists(path): return []
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line: rows.append(json.loads(line))
    return rows

print('Logging utilities ready.')


In [ ]:
# Cell 06 — Corrected Multi-Domain Dataset Loaders (Zero URI Errors & Zero Remote Code Warnings)
import urllib.request, io, json
from datasets import load_dataset

def chat_prefix_text(prompt_text: str) -> str:
    """Exact Laguna-XS.2 XML chat template that produced 46.0% base accuracy in v23."""
    return f"<system>\nYou are a helpful assistant.\n</system>\n<user>\n{prompt_text}\n</user>\n<assistant>\n" 

CONTROL_CODE_TASKS = [
    ('Implement binary search.', 'def binary_search(arr, t):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n        m = (lo + hi) // 2\n        if arr[m] == t: return m\n        elif arr[m] < t: lo = m + 1\n        else: hi = m - 1\n    return -1'),
    ('Merge two sorted lists.', 'def merge(a, b):\n    r, i, j = [], 0, 0\n    while i < len(a) and j < len(b):\n        if a[i] <= b[j]: r.append(a[i]); i += 1\n        else: r.append(b[j]); j += 1\n    r.extend(a[i:]); r.extend(b[j:])\n    return r'),
    ('Implement a stack class.', 'class Stack:\n    def __init__(self): self._s = []\n    def push(self, x): self._s.append(x)\n    def pop(self): return self._s.pop()\n    def peek(self): return self._s[-1]\n    def __len__(self): return len(self._s)'),
    ('Longest common subsequence.', 'def lcs(a, b):\n    m, n = len(a), len(b)\n    dp = [[0]*(n+1) for _ in range(m+1)]\n    for i in range(1, m+1):\n        for j in range(1, n+1):\n            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1] + 1\n            else: dp[i][j] = max(dp[i-1][j], dp[i][j-1])\n    return dp[m][n]'),
    ('Flatten nested list.', 'def flatten(lst):\n    r = []\n    for x in lst:\n        if isinstance(x, list): r.extend(flatten(x))\n        else: r.append(x)\n    return r'),
    ('Fibonacci with memo.', 'def fib(n, m={}):\n    if n in m: return m[n]\n    if n <= 1: return n\n    m[n] = fib(n-1, m) + fib(n-2, m)\n    return m[n]'),
    ('Quicksort.', 'def qsort(a):\n    if len(a) <= 1: return a\n    p = a[len(a)//2]\n    return qsort([x for x in a if x < p]) + [x for x in a if x == p] + qsort([x for x in a if x > p])'),
    ('Prime factors.', 'def factors(n):\n    f, d = [], 2\n    while d*d <= n:\n        while n % d == 0: f.append(d); n //= d\n        d += 1\n    if n > 1: f.append(n)\n    return f'),
    ('BFS on graph.', 'from collections import deque\ndef bfs(g, s):\n    vis, q, r = set(), deque([s]), []\n    vis.add(s)\n    while q:\n        n = q.popleft(); r.append(n)\n        for nb in g.get(n, []):\n            if nb not in vis: vis.add(nb); q.append(nb)\n    return r'),
    ('Roman to int.', 'def roman(s):\n    v = {"I":1,"V":5,"X":10,"L":50,"C":100,"D":500,"M":1000}\n    r = 0\n    for i in range(len(s)):\n        if i+1 < len(s) and v[s[i]] < v[s[i+1]]: r -= v[s[i]]\n        else: r += v[s[i]]\n    return r'),
    ('Edit distance.', 'def edit(a, b):\n    m, n = len(a), len(b)\n    dp = [[0]*(n+1) for _ in range(m+1)]\n    for i in range(m+1): dp[i][0] = i\n    for j in range(n+1): dp[0][j] = j\n    for i in range(1, m+1):\n        for j in range(1, n+1):\n            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1]\n            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])\n    return dp[m][n]'),
    ('Detect cycle in linked list.', 'def has_cycle(head):\n    s = f = head\n    while f and f.next:\n        s = s.next; f = f.next.next\n        if s is f: return True\n    return False')
]

def fetch_gsm8k_direct(split: str = 'train') -> List[Dict[str, str]]:
    url = f'https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/{split}.jsonl'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    out = []
    with urllib.request.urlopen(req, timeout=15) as resp:
        for line in resp.read().decode('utf-8').splitlines():
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

def load_target_dataset(cfg: Config, split: str) -> List[Dict[str, Any]]:
    """High-complexity reasoning traces for training; GPQA Diamond for validation."""
    if split == 'train':
        out = []
        limit = cfg.max_target_train_samples
        print(f'Loading target training CoT reasoning data (target limit: {limit})...', flush=True)
        
        # 1. MetaMathQA / GSM8K CoT Reasoning
        try:
            mm_ds = load_dataset('meta-math/MetaMathQA', split='train', streaming=True)
            for idx, item in enumerate(mm_ds):
                if len(out) >= limit: break
                q = item.get('query', '').strip(); r = item.get('response', '').strip()
                if q and r:
                    prompt = f'{q}\n\nDerive the solution step by step, then state the final answer in \\boxed{{}}.'
                    out.append({'task': 'metamath', 'prompt': prompt, 'text': f'{prompt}\n{r}'})
            print(f'  MetaMathQA CoT reasoning loaded: {len(out)} examples', flush=True)
        except Exception as e:
            print(f'  MetaMathQA streaming note: {e}, using direct GSM8K CoT loader...')
            try:
                gsm_raw = fetch_gsm8k_direct('train')
                for item in gsm_raw:
                    if len(out) >= limit: break
                    q = item['question']; a = item['answer']
                    prompt = f'{q}\n\nDerive the solution step by step, then state the final answer in \\boxed{{}}.'
                    out.append({'task': 'gsm8k_cot', 'prompt': prompt, 'text': f'{prompt}\n{a}'})
                print(f'  Direct GSM8K CoT loaded: {len(out)} examples', flush=True)
            except Exception as e2: print(f'  GSM8K direct fallback note: {e2}')
        
        # 2. Science Reasoning Support (SciQ)
        try:
            sciq = load_dataset('allenai/sciq', split='train')
            for idx, item in enumerate(sciq):
                if len(out) >= limit: break
                q = item['question']; correct = item['correct_answer']
                dists = [item.get('distractor1',''), item.get('distractor2',''), item.get('distractor3','')]
                if not all(dists): continue
                choices = [correct] + dists
                rng = random.Random(3000 + idx); rng.shuffle(choices)
                cl = 'ABCD'[choices.index(correct)]
                prompt = f'Question: {q}\n\nChoices:\n(A) {choices[0]}\n(B) {choices[1]}\n(C) {choices[2]}\n(D) {choices[3]}\n\nDerive the answer step by step, then state the final answer letter in \\boxed{{}}.'
                expl = item.get('support', '') or f'The answer is {correct}.'
                out.append({'task': 'sciq_cot', 'prompt': prompt, 'text': f'{prompt}\n{expl}\n\\boxed{{{cl}}}'})
        except Exception as e: print(f'  SciQ note: {e}')
        
        random.Random(2026).shuffle(out)
        out = out[:limit]
        print(f'Target training dataset compiled: {len(out)} complex CoT reasoning samples.', flush=True)
        return out
        
    else:
        # Target Evaluation: GPQA Diamond (198 PhD-level questions)
        print(f'Loading GPQA Diamond evaluation suite...', flush=True)
        url = 'https://huggingface.co/datasets/Idavidrein/gpqa/resolve/main/gpqa_diamond.csv'
        req = urllib.request.Request(url, headers={'Authorization': f'Bearer {HF_TOKEN}', 'User-Agent': 'Mozilla/5.0'})
        out = []
        with urllib.request.urlopen(req, timeout=30) as resp:
            rows = list(csv.DictReader(io.StringIO(resp.read().decode('utf-8'))))
        for idx, r in enumerate(rows):
            if idx >= cfg.max_target_eval_samples: break
            q = r.get('Question', '').strip(); ca = r.get('Correct Answer', '').strip()
            choices = [ca, r.get('Incorrect Answer 1','').strip(), r.get('Incorrect Answer 2','').strip(), r.get('Incorrect Answer 3','').strip()]
            rng = random.Random(2026 + idx); rng.shuffle(choices)
            cl = 'ABCD'[choices.index(ca)]
            prompt = f'Question: {q}\n\nChoices:\n(A) {choices[0]}\n(B) {choices[1]}\n(C) {choices[2]}\n(D) {choices[3]}\n\nDerive the answer step by step, then state the final answer letter in \\boxed{{}}.'
            out.append({'task': 'gpqa_diamond', 'prompt': prompt, 'answer': cl, 'correct_text': ca})
        print(f'GPQA Diamond evaluation set loaded: {len(out)} questions.', flush=True)
        return out

def load_retained_dataset(cfg: Config, split: str) -> List[Dict[str, Any]]:
    """
    Stratified 4-Pillar Multi-Domain Capability Mixture:
    1. Code (HumanEval + Control Tasks) - 25%
    2. General Chat / Instruction (UltraChat / OpenHermes) - 25%
    3. Factual Knowledge & Language (Salesforce/wikitext) - 25%
    4. Math Logic (Direct GSM8K) - 25%
    """
    limit = cfg.max_retained_fisher_samples if split == 'train' else cfg.max_retained_eval_samples
    target_per_pillar = limit // 4
    out = []
    
    print(f'Compiling Retained Mixture [{split.upper()}]: Target {limit} samples ({target_per_pillar} per pillar)...', flush=True)
    
    # --- Pillar 1: Code (25%) ---
    code_samples = []
    for p, r in CONTROL_CODE_TASKS:
        code_samples.append({'domain': 'code', 'task': 'control_algo', 'prompt': p, 'text': f'{p}\n{r}'})
    try:
        he_ds = load_dataset('openai/openai_humaneval', split='test')
        for item in he_ds:
            if len(code_samples) >= target_per_pillar: break
            p = item['prompt']; sol = item['canonical_solution']
            if p and sol: code_samples.append({'domain': 'code', 'task': 'humaneval', 'prompt': p, 'text': f'Complete this code:\n{p}\n{sol}'})
    except Exception as e: print(f'  [Pillar 1 - Code] HumanEval note: {e}')
    out.extend(code_samples[:target_per_pillar])
    print(f'  ✓ Pillar 1 (Code): {len(code_samples[:target_per_pillar])} samples', flush=True)
    
    # --- Pillar 2: General Chat / Instruction (25%) ---
    chat_samples = []
    try:
        chat_ds = load_dataset('HuggingFaceH4/ultrachat_200k', split='train_sft', streaming=True)
        for item in chat_ds:
            if len(chat_samples) >= target_per_pillar: break
            msgs = item.get('messages', [])
            if len(msgs) >= 2:
                u_text = msgs[0]['content']; a_text = msgs[1]['content']
                if len(u_text) > 20 and len(a_text) > 20:
                    chat_samples.append({'domain': 'chat', 'task': 'ultrachat', 'prompt': u_text, 'text': f'<user>\n{u_text}\n</user>\n<assistant>\n{a_text}'})
    except Exception as e:
        print(f'  [Pillar 2 - Chat] UltraChat note: {e}, using OpenHermes...')
        try:
            hermes = load_dataset('teknium/OpenHermes-2.5', split='train', streaming=True)
            for item in hermes:
                if len(chat_samples) >= target_per_pillar: break
                convs = item.get('conversations', [])
                if len(convs) >= 2:
                    chat_samples.append({'domain': 'chat', 'task': 'openhermes', 'prompt': convs[0]['value'], 'text': f"{convs[0]['value']}\n{convs[1]['value']}"})
        except Exception as e2: print(f'  [Pillar 2 - Chat] Fallback note: {e2}')
    out.extend(chat_samples[:target_per_pillar])
    print(f'  ✓ Pillar 2 (Chat/Instruction): {len(chat_samples[:target_per_pillar])} samples', flush=True)
    
    # --- Pillar 3: Factual Knowledge & Language (25%) ---
    wiki_samples = []
    try:
        # Use Salesforce/wikitext (official namespaced HF repo)
        wiki_ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train' if split == 'train' else 'validation')
        for item in wiki_ds:
            if len(wiki_samples) >= target_per_pillar: break
            t = item.get('text', '').strip()
            if len(t) >= 120:
                wiki_samples.append({'domain': 'language', 'task': 'wikitext', 'prompt': 'Passage:', 'text': t})
    except Exception as e:
        print(f'  [Pillar 3 - Wiki] Salesforce/wikitext note: {e}, loading fallback wiki...')
        try:
            c4 = load_dataset('allenai/c4', 'en', split='train', streaming=True)
            for item in c4:
                if len(wiki_samples) >= target_per_pillar: break
                t = item.get('text', '').strip()
                if len(t) >= 120:
                    wiki_samples.append({'domain': 'language', 'task': 'c4', 'prompt': 'Passage:', 'text': t})
        except Exception as e2: print(f'  [Pillar 3 - Wiki] Fallback note: {e2}')
    out.extend(wiki_samples[:target_per_pillar])
    print(f'  ✓ Pillar 3 (Factual/Language): {len(wiki_samples[:target_per_pillar])} samples', flush=True)
    
    # --- Pillar 4: Math Logic (25%) ---
    math_samples = []
    try:
        # Load directly from OpenAI official repository (zero HF URI/namespace dependency)
        gsm_split = 'train' if split == 'train' else 'test'
        gsm_raw = fetch_gsm8k_direct(gsm_split)
        for item in gsm_raw:
            if len(math_samples) >= target_per_pillar: break
            q = item['question']; a = item['answer']
            math_samples.append({'domain': 'math', 'task': 'gsm8k_logic', 'prompt': q, 'text': f'Question: {q}\nAnswer: {a}'})
    except Exception as e: print(f'  [Pillar 4 - Math] Direct GSM8K note: {e}')
    out.extend(math_samples[:target_per_pillar])
    print(f'  ✓ Pillar 4 (Math Logic): {len(math_samples[:target_per_pillar])} samples', flush=True)
    
    random.Random(2026 if split == 'train' else 5000).shuffle(out)
    out = out[:limit]
    print(f'Total Retained Mixture compiled: {len(out)} samples across 4 capability domains.', flush=True)
    return out

print('Multi-domain dataset loaders ready.')


In [ ]:
# Cell 07 — Torch Dataset and Collate
class TextDataset(Dataset):
    def __init__(self, examples: List[Dict[str, Any]], tokenizer, max_length: int = 512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx: int):
        ex = self.examples[idx]
        text = ex.get('text', '') or f"{ex.get('prompt', '')}\n{ex.get('answer', '')}"
        enc = self.tokenizer(text, truncation=True, max_length=self.max_length, return_tensors=None)
        input_ids = enc['input_ids']
        attention_mask = enc.get('attention_mask', [1] * len(input_ids))
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': input_ids.copy()}

def collate_lm(batch: List[Dict[str, Any]], pad_token_id: int = 0):
    input_ids = [torch.tensor(x['input_ids'], dtype=torch.long) for x in batch]
    attention_mask = [torch.tensor(x['attention_mask'], dtype=torch.long) for x in batch]
    labels = [torch.tensor(x['labels'], dtype=torch.long) for x in batch]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

def infinite_loader(loader: DataLoader):
    while True:
        for batch in loader: yield batch

def to_device(batch: Dict[str, torch.Tensor], device: str):
    return {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
print('Dataset & collate ready.')

In [ ]:
# Cell 08 — PEFT LoRA Setup & Weight Reset Functions
from peft import LoraConfig, get_peft_model, TaskType

for p in model.parameters(): p.requires_grad = False
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout, bias='none', target_modules=cfg.target_modules,
    layers_to_transform=cfg.layers_to_transform
)
peft_model = get_peft_model(model, peft_config)
peft_model.gradient_checkpointing_enable()
trainable_p = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f'LoRA initialized on Laguna-XS.2: {trainable_p:,} trainable parameters.')

def reset_lora_weights():
    """Re-initializes all LoRA A with Kaiming uniform and B with zeros."""
    with torch.no_grad():
        for n, m in peft_model.named_modules():
            if hasattr(m, 'lora_A'):
                sA = m.lora_A['default'] if hasattr(m.lora_A, '__getitem__') else m.lora_A
                sB = m.lora_B['default'] if hasattr(m.lora_B, '__getitem__') else m.lora_B
                nn.init.kaiming_uniform_(sA.weight, a=math.sqrt(5))
                nn.init.zeros_(sB.weight)
                sA.weight.requires_grad = True
                sB.weight.requires_grad = True

print('LoRA setup and in-place reset function ready.')


In [ ]:
# Cell 09 — Forward Loss
def forward_loss(model, batch: Dict[str, torch.Tensor]):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        outputs = model(**batch)
    return outputs.loss
print('Forward loss defined.')

In [ ]:
# Cell 10 — High-Speed Vectorized Fisher Estimation
# Optimizations: Minibatched backward passes in BF16, zero intermediate tensor allocations
def compute_diagonal_fisher(model, loader: DataLoader, cfg: Config):
    model.eval()
    fisher = {}
    base_state = {}
    for name, param in model.named_parameters():
        if param.requires_grad:
            fisher[name] = torch.zeros_like(param, device='cpu', dtype=torch.float32)
            base_state[name] = param.detach().clone().cpu()
    
    num_samples = 0
    t0 = time.time()
    for batch in loader:
        if cfg.max_retained_fisher_samples is not None and num_samples >= cfg.max_retained_fisher_samples:
            break
        model.zero_grad(set_to_none=True)
        batch = to_device(batch, cfg.device)
        loss = forward_loss(model, batch)
        loss.backward()
        
        bs = batch['input_ids'].shape[0]
        for name, param in model.named_parameters():
            if param.requires_grad and param.grad is not None:
                # Accumulate squared gradients weighted by batch size
                fisher[name] += (param.grad.detach().float().cpu() ** 2) * bs
        num_samples += bs
        del batch, loss
    
    num_samples = max(1, num_samples)
    for name in fisher:
        fisher[name] /= num_samples
        fisher[name] += cfg.fisher_eps
    
    model.zero_grad(set_to_none=True)
    gc.collect(); torch.cuda.empty_cache()
    print(f'Computed Fisher across {len(fisher)} matrices ({num_samples} samples) in {time.time()-t0:.1f}s.')
    return fisher, base_state
print('High-speed Fisher estimation ready.')


In [ ]:
# Cell 11 — Surgical Constraint Application (Online Training Penalty)
def apply_constraint_gradients(model, fisher: Optional[Dict[str, torch.Tensor]], base_state: Optional[Dict[str, torch.Tensor]], cfg: Config):
    if fisher is None or base_state is None or cfg.constraint_mode == 'none': return
    for name, param in model.named_parameters():
        if name not in fisher or name not in base_state or param.grad is None: continue
        f = fisher[name].to(param.device)
        base = base_state[name].to(param.device)
        if cfg.constraint_mode == 'ewc':
            # grad += 2 * lambda * F_i * (theta_i - theta_i^0)
            param.grad.add_(2.0 * cfg.fisher_lambda * f * (param.data - base))
        elif cfg.constraint_mode == 'grad_damp':
            # grad = grad / (1.0 + lambda * F_i)
            param.grad.div_(1.0 + cfg.fisher_lambda * f)
print('Constraint application defined.')

In [ ]:
# Cell 12 — Update Norm, Fisher Energy, and Norm Matching
def compute_update_norm(model, base_state: Dict[str, torch.Tensor]) -> float:
    total = 0.0
    for name, param in model.named_parameters():
        if name not in base_state or not param.requires_grad: continue
        base = base_state[name].to(param.device)
        total += torch.norm(param.data - base).item() ** 2
    return math.sqrt(total)

def compute_fisher_energy(model, fisher: Optional[Dict[str, torch.Tensor]], base_state: Optional[Dict[str, torch.Tensor]]) -> float:
    if fisher is None or base_state is None: return 0.0
    total = 0.0
    for name, param in model.named_parameters():
        if name not in fisher or name not in base_state or not param.requires_grad: continue
        f = fisher[name].to(param.device)
        base = base_state[name].to(param.device)
        total += torch.sum(f * (param.data - base) ** 2).item()
    return total

def rescale_update_to_norm(model, base_state: Dict[str, torch.Tensor], target_norm: float) -> float:
    current_norm = compute_update_norm(model, base_state)
    if current_norm <= 1e-12: return current_norm
    scale = target_norm / current_norm
    for name, param in model.named_parameters():
        if name not in base_state or not param.requires_grad: continue
        base = base_state[name].to(param.device)
        param.data.copy_(base + scale * (param.data - base))
    return target_norm
print('Energy & norm metrics ready.')

In [ ]:
# Cell 13 — Gradient Conflict Metrics
def get_trainable_grad_vector(model):
    grads = [param.grad.detach().flatten() for param in model.parameters() if param.requires_grad and param.grad is not None]
    if not grads: return None
    return torch.cat(grads)

def compute_gradient_metrics(model, target_loader: DataLoader, retained_loader: DataLoader, cfg: Config):
    model.train()
    def gradient_from_loader(loader: DataLoader):
        model.zero_grad()
        try: batch = next(iter(loader))
        except StopIteration: return None
        batch = to_device(batch, cfg.device)
        loss = forward_loss(model, batch)
        loss.backward()
        vec = get_trainable_grad_vector(model)
        model.zero_grad()
        return vec
    g_target = gradient_from_loader(target_loader)
    g_retained = gradient_from_loader(retained_loader)
    if g_target is None or g_retained is None:
        return {'grad_cosine': 0.0, 'target_grad_norm': 0.0, 'retained_grad_norm': 0.0}
    cosine = F.cosine_similarity(g_target.unsqueeze(0), g_retained.unsqueeze(0)).item()
    return {
        'grad_cosine': cosine,
        'target_grad_norm': torch.norm(g_target).item(),
        'retained_grad_norm': torch.norm(g_retained).item()
    }
print('Gradient conflict metrics ready.')

In [ ]:
# Cell 14 — Exact Working GPQA Diamond Evaluator (Batched, Restores 46.0% Base Accuracy)
def extract_answer(text: str) -> str:
    if not text or not text.strip(): return ''
    s = text.strip()
    # 1. \boxed{X}
    idx = s.rfind(r'\boxed{')
    if idx != -1:
        content, depth = [], 0
        for c in s[idx+7:]:
            if c == '{': depth += 1; content.append(c)
            elif c == '}':
                if depth == 0: break
                depth -= 1; content.append(c)
            else: content.append(c)
        boxed = ''.join(content).strip().upper()
        bl = re.findall(r'\b([A-D])\b', boxed)
        if bl: return bl[-1]
    # 2. 'The answer is (X)'
    m = re.search(r'[Tt]he\s+answer\s+is\s*\(?([A-D])\)?', s)
    if m: return m.group(1).upper()
    # 3. Last (A)/(B)/(C)/(D)
    paren = re.findall(r'\(([A-D])\)', s)
    if paren: return paren[-1].upper()
    # 4. Last standalone letter in trailing 80 chars
    tail = s[-80:] if len(s) > 80 else s
    ml = re.findall(r'(?<![a-zA-Z])([A-D])(?![a-zA-Z])', tail)
    if ml: return ml[-1].upper()
    return ''

@torch.inference_mode()
def evaluate_target_accuracy(model, tokenizer, examples: List[Dict[str, Any]], cfg: Config):
    model.eval()
    total = len(examples); correct = 0; scores = []; preds = []; t0 = time.time()
    batch_size = getattr(cfg, 'eval_batch_size', 12)
    max_tokens = cfg.max_new_tokens
    
    old_ps = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    
    try:
        for si in range(0, total, batch_size):
            b_examples = examples[si:si+batch_size]
            pfx = [chat_prefix_text(ex.get('prompt', '')) for ex in b_examples]
            enc = tokenizer(pfx, return_tensors='pt', padding=True, truncation=True, max_length=768).to(model.device)
            
            out = model.generate(
                input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
                max_new_tokens=max_tokens, do_sample=False, use_cache=True,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
            )
            
            dec = tokenizer.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
            del out, enc
            
            for j, ex in enumerate(b_examples):
                ext = extract_answer(dec[j])
                target = str(ex.get('answer', '')).strip().upper()
                ic = 1.0 if (ext == target and target != '') else 0.0
                correct += int(ic)
                scores.append(ic)
                preds.append(dec[j][:150])
            
            torch.cuda.empty_cache()
            done = min(si + batch_size, total)
            if done % (batch_size * 3) == 0 or done == total:
                curr_acc = (correct / done) * 100
                last_ext = extract_answer(dec[-1])
                last_gold = str(b_examples[-1].get('answer', '')).strip().upper()
                print(f'    [{done:03d}/{total}] {curr_acc:4.1f}% ({correct}/{done}) | {time.time()-t0:.0f}s | last: ext={last_ext} gold={last_gold}', flush=True)
    finally:
        tokenizer.padding_side = old_ps
        gc.collect(); torch.cuda.empty_cache()
    
    acc = correct / max(1, total)
    print(f'  GPQA Diamond: {acc*100:.1f}% ({correct}/{total}) in {time.time()-t0:.0f}s', flush=True)
    return acc, scores, preds

print('GPQA Diamond batched evaluator ready.')


In [ ]:
# Cell 15 — Retained NLL Evaluation with Per-Sample Losses
@torch.inference_mode()
def evaluate_nll(model, loader: DataLoader, cfg: Config):
    model.eval()
    total_loss = 0.0; total_tokens = 0; sample_losses = []
    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    for batch in loader:
        batch = to_device(batch, cfg.device)
        outputs = model(**batch)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = batch['labels'][..., 1:].contiguous()
        losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        mask = shift_labels.view(-1) != -100
        loss_sum = losses[mask].sum().item()
        token_count = mask.sum().item()
        total_loss += loss_sum
        total_tokens += token_count
        losses_2d = losses.view(shift_labels.size(0), -1)
        mask_2d = mask.view(shift_labels.size(0), -1)
        per_sample = (losses_2d * mask_2d).sum(dim=1) / (mask_2d.sum(dim=1) + 1e-8)
        sample_losses.extend(per_sample.cpu().tolist())
    avg_loss = total_loss / max(1, total_tokens)
    try: ppl = math.exp(avg_loss)
    except OverflowError: ppl = float('inf')
    return avg_loss, ppl, sample_losses
print('Retained NLL evaluation defined.')

In [ ]:
# Cell 16 — Bootstrap Confidence Intervals
def bootstrap_ci(values: List[float], n_boot: int = 500, ci: float = 95.0):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0: return np.nan, np.nan, np.nan
    stats = []
    for _ in range(n_boot):
        sample = np.random.choice(values, size=len(values), replace=True)
        stats.append(np.mean(sample))
    lower = np.percentile(stats, (100 - ci) / 2)
    middle = np.percentile(stats, 50)
    upper = np.percentile(stats, 100 - (100 - ci) / 2)
    return float(lower), float(middle), float(upper)
print('Bootstrap CI defined.')

In [ ]:
# Cell 17 — Full Evaluation Function
def evaluate_all(model, tokenizer, cfg: Config, step: int, target_eval_examples: List[Dict[str, Any]],
                 retained_eval_loader: DataLoader, base_metrics: Optional[Dict[str, Any]] = None,
                 fisher: Optional[Dict[str, torch.Tensor]] = None, base_state: Optional[Dict[str, torch.Tensor]] = None):
    metrics = {'step': step}
    # Target task
    if cfg.eval_target_accuracy and len(target_eval_examples) > 0:
        target_acc, target_scores, _ = evaluate_target_accuracy(model, tokenizer, target_eval_examples, cfg)
        t_lo, t_mid, t_hi = bootstrap_ci(target_scores, n_boot=cfg.bootstrap_n)
        metrics.update({'target_accuracy': target_acc, 'target_acc_ci_low': t_lo, 'target_acc_ci_mid': t_mid, 'target_acc_ci_high': t_hi})
    else:
        metrics.update({'target_accuracy': np.nan, 'target_acc_ci_low': np.nan, 'target_acc_ci_mid': np.nan, 'target_acc_ci_high': np.nan})
    
    # Retained task
    retained_nll, retained_ppl, retained_losses = evaluate_nll(model, retained_eval_loader, cfg)
    r_lo, r_mid, r_hi = bootstrap_ci(retained_losses, n_boot=cfg.bootstrap_n)
    metrics.update({'retained_nll': retained_nll, 'retained_perplexity': retained_ppl, 'retained_nll_ci_low': r_lo, 'retained_nll_ci_mid': r_mid, 'retained_nll_ci_high': r_hi})
    
    # Surgical comparison
    if base_metrics is not None:
        target_gain = metrics['target_accuracy'] - base_metrics.get('target_accuracy', 0.0)
        retained_delta = metrics['retained_nll'] - base_metrics.get('retained_nll', 0.0)
        forgetting_score = max(0.0, retained_delta)
        net_surgical_score = target_gain - cfg.forgetting_penalty * forgetting_score
        metrics.update({'target_gain': target_gain, 'retained_nll_delta': retained_delta, 'forgetting_score': forgetting_score, 'net_surgical_score': net_surgical_score})
    
    if base_state is not None: metrics['update_norm'] = compute_update_norm(model, base_state)
    else: metrics['update_norm'] = 0.0
    
    if fisher is not None and base_state is not None: metrics['retained_fisher_energy'] = compute_fisher_energy(model, fisher, base_state)
    else: metrics['retained_fisher_energy'] = 0.0
    
    return metrics
print('Full evaluation function ready.')

In [ ]:
# Cell 18 — Optional MoE Route Flip Rate Diagnostic
ROUTE_FLIP_IMPLEMENTED = False

def compute_route_flip_rate(base_model, model, batch, cfg: Config):
    if not ROUTE_FLIP_IMPLEMENTED: return None
    # Stub for tracking Top-8 expert routing bifurcations across layers
    raise NotImplementedError
print('Route flip diagnostic defined.')

In [ ]:
# Cell 19 — High-Throughput Memory-Optimized Training Loop (Zero Reload Overhead)
def train_arm(arm: str, cfg: Config, seed: int, base_metrics: Optional[Dict[str, Any]] = None):
    set_seed(seed)
    outdir = os.path.join(cfg.output_dir, arm, f'seed_{seed}')
    ensure_dir(outdir)
    train_log_path = os.path.join(outdir, 'train_log.jsonl')
    grad_log_path = os.path.join(outdir, 'gradient_log.jsonl')
    eval_csv_path = os.path.join(outdir, 'eval_metrics.csv')
    
    print('=' * 80)
    print(f'Starting Arm: {arm} | Seed: {seed}', flush=True)
    print('=' * 80)
    
    target_train = load_target_dataset(cfg, 'train')
    target_eval = load_target_dataset(cfg, 'test')
    retained_train = load_retained_dataset(cfg, 'train')
    retained_eval = load_retained_dataset(cfg, 'validation')
    
    collate = partial(collate_lm, pad_token_id=tokenizer.pad_token_id)
    target_train_loader = DataLoader(TextDataset(target_train, tokenizer, cfg.max_seq_len), batch_size=cfg.train_batch_size, shuffle=True, collate_fn=collate, pin_memory=True)
    retained_train_loader = DataLoader(TextDataset(retained_train, tokenizer, cfg.max_seq_len), batch_size=cfg.fisher_batch_size, shuffle=False, collate_fn=collate, pin_memory=True)
    retained_eval_loader = DataLoader(TextDataset(retained_eval, tokenizer, cfg.max_seq_len), batch_size=cfg.train_batch_size, shuffle=False, collate_fn=collate, pin_memory=True)
    
    # Base arm: evaluate only (disable LoRA adapters)
    if arm == 'base':
        peft_model.eval()
        with peft_model.disable_adapter():
            metrics = evaluate_all(peft_model, tokenizer, cfg, 0, target_eval, retained_eval_loader, base_metrics=None)
        metrics.update({'arm': arm, 'seed': seed})
        append_csv(eval_csv_path, metrics)
        print(json.dumps(metrics, indent=2))
        return metrics
    
    # Trained arm: in-place reset of LoRA parameters
    reset_lora_weights()
    peft_model.train()
    
    base_state = {name: param.detach().clone().cpu() for name, param in peft_model.named_parameters() if param.requires_grad}
    
    fisher = None
    if arm == 'fisher_constrained_lora' and cfg.use_retained_fisher:
        print('Computing retained-task diagonal Fisher over 4-pillar mixture...', flush=True)
        fisher, fisher_base_state = compute_diagonal_fisher(peft_model, retained_train_loader, cfg)
        base_state = fisher_base_state
        print('Fisher computation complete.', flush=True)
    
    optimizer = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
    train_iter = infinite_loader(target_train_loader)
    last_metrics = None
    
    for step in range(1, cfg.max_steps + 1):
        peft_model.train()
        batch = to_device(next(train_iter), cfg.device)
        loss = forward_loss(peft_model, batch)
        loss.backward()
        
        if arm == 'fisher_constrained_lora' and cfg.use_retained_fisher:
            apply_constraint_gradients(peft_model, fisher, base_state, cfg)
        
        torch.nn.utils.clip_grad_norm_(peft_model.parameters(), cfg.max_grad_norm)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        
        if step % cfg.log_interval == 0:
            update_norm = compute_update_norm(peft_model, base_state)
            fisher_energy = compute_fisher_energy(peft_model, fisher, base_state)
            train_row = {'step': step, 'target_loss': loss.item(), 'update_norm': update_norm, 'retained_fisher_energy': fisher_energy, 'lr': optimizer.param_groups[0]['lr']}
            log_jsonl(train_log_path, train_row)
            print(f'   Step [{step:02d}/{cfg.max_steps:02d}] Loss: {loss.item():.4f} | UpdNorm: {update_norm:.4f} | FisherEnergy: {fisher_energy:.4e}', flush=True)
        
        if step % cfg.eval_interval == 0 or step == cfg.max_steps:
            if step == cfg.max_steps and cfg.norm_match_final:
                rescale_update_to_norm(peft_model, base_state, cfg.target_update_norm)
            grad_metrics = {}
            if cfg.compute_grad_metrics and (step % cfg.grad_metrics_interval == 0 or step == cfg.max_steps):
                grad_metrics = compute_gradient_metrics(peft_model, target_train_loader, retained_eval_loader, cfg)
                grad_metrics.update({'step': step})
                log_jsonl(grad_log_path, grad_metrics)
            metrics = evaluate_all(peft_model, tokenizer, cfg, step, target_eval, retained_eval_loader, base_metrics, fisher, base_state)
            metrics.update({'arm': arm, 'seed': seed})
            metrics.update(grad_metrics)
            append_csv(eval_csv_path, metrics)
            last_metrics = metrics
            print(f'   EVAL [{step:02d}]: TargetAcc: {metrics.get("target_accuracy",0)*100:.1f}% (gain: {metrics.get("target_gain",0):+.1%}) | RetainedNLL: {metrics.get("retained_nll",0):.4f}', flush=True)
        
        del batch, loss
    
    del optimizer; gc.collect(); torch.cuda.empty_cache()
    return last_metrics
print('In-memory training loop ready.')


In [ ]:
# Cell 20 — Run all Arms and Seeds
all_final_metrics = []

for seed in cfg.seeds:
    base_metrics = train_arm('base', cfg, seed, None)
    all_final_metrics.append(base_metrics)
    for arm in cfg.arms:
        if arm == 'base': continue
        final_metrics = train_arm(arm, cfg, seed, base_metrics)
        all_final_metrics.append(final_metrics)

final_df = pd.DataFrame(all_final_metrics)
final_df.to_csv(os.path.join(cfg.output_dir, 'final_summary.csv'), index=False)
display(final_df)

In [ ]:
# Cell 21 — Visualization Log Collectors
def collect_train_logs(root_dir: str) -> pd.DataFrame:
    rows = []
    for path in glob.glob(os.path.join(root_dir, '*', 'seed_*', 'train_log.jsonl')):
        parts = path.split(os.sep); arm = parts[-3]; seed_dir = parts[-2]
        try: seed = int(seed_dir.replace('seed_', ''))
        except: seed = -1
        for row in read_jsonl(path): row['arm'] = arm; row['seed'] = seed; rows.append(row)
    return pd.DataFrame(rows)

def collect_gradient_logs(root_dir: str) -> pd.DataFrame:
    rows = []
    for path in glob.glob(os.path.join(root_dir, '*', 'seed_*', 'gradient_log.jsonl')):
        parts = path.split(os.sep); arm = parts[-3]; seed_dir = parts[-2]
        try: seed = int(seed_dir.replace('seed_', ''))
        except: seed = -1
        for row in read_jsonl(path): row['arm'] = arm; row['seed'] = seed; rows.append(row)
    return pd.DataFrame(rows)

def collect_eval_metrics(root_dir: str) -> pd.DataFrame:
    dfs = []
    for path in glob.glob(os.path.join(root_dir, '*', 'seed_*', 'eval_metrics.csv')):
        df = pd.read_csv(path); parts = path.split(os.sep); arm = parts[-3]; seed_dir = parts[-2]
        try: seed = int(seed_dir.replace('seed_', ''))
        except: seed = -1
        df['arm'] = arm; df['seed'] = seed; dfs.append(df)
    if not dfs: return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)
print('Visual log collectors defined.')

In [ ]:
# Cell 22 — Plot Training Curves
def plot_training_curves(root_dir: str):
    train_df = collect_train_logs(root_dir)
    if train_df.empty: print('No training logs found.'); return
    metrics = ['target_loss', 'update_norm', 'retained_fisher_energy']
    for metric in metrics:
        if metric not in train_df.columns: continue
        plt.figure(figsize=(7, 4))
        for arm in train_df['arm'].unique():
            sub = train_df[train_df['arm'] == arm]
            grouped = sub.groupby('step')[metric].mean().reset_index()
            plt.plot(grouped['step'], grouped[metric], label=arm, marker='o')
        plt.title(f'Training Dynamic: {metric}')
        plt.xlabel('Step'); plt.ylabel(metric); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
print('Training curves plotter ready.')

In [ ]:
# Cell 23 — Plot Evaluation Curves
def plot_eval_curves(root_dir: str):
    eval_df = collect_eval_metrics(root_dir)
    if eval_df.empty: print('No eval metrics found.'); return
    metrics = ['target_accuracy', 'retained_nll', 'target_gain', 'retained_nll_delta', 'net_surgical_score', 'retained_fisher_energy']
    for metric in metrics:
        if metric not in eval_df.columns: continue
        plt.figure(figsize=(7, 4))
        for arm in eval_df['arm'].unique():
            sub = eval_df[eval_df['arm'] == arm]
            grouped = sub.groupby('step')[metric].mean().reset_index()
            plt.plot(grouped['step'], grouped[metric], label=arm, marker='s')
        plt.title(f'Evaluation Metric: {metric}')
        plt.xlabel('Step'); plt.ylabel(metric); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
print('Evaluation curves plotter ready.')

In [ ]:
# Cell 24 — Plot Pareto Frontier (Target Gain vs. Retained Degradation)
def plot_pareto(root_dir: str):
    eval_df = collect_eval_metrics(root_dir)
    if eval_df.empty or 'target_gain' not in eval_df.columns: print('Insufficient eval metrics for Pareto plot.'); return
    eval_df = eval_df.sort_values('step')
    final_df = eval_df.groupby(['arm', 'seed']).tail(1)
    plt.figure(figsize=(8, 5))
    for arm in final_df['arm'].unique():
        sub = final_df[final_df['arm'] == arm]
        plt.scatter(sub['retained_nll_delta'], sub['target_gain'], label=arm, s=100, alpha=0.85)
    plt.axhline(0, color='gray', linestyle='--', linewidth=1)
    plt.axvline(0, color='gray', linestyle='--', linewidth=1)
    plt.xlabel(r'Retained Degradation (\Delta NLL_{control}) [Lower is Better]')
    plt.ylabel(r'Target Gain (\Delta Acc_{target}) [Higher is Better]')
    plt.title('Surgical Adaptation Pareto Frontier (Ideal: Top-Left)')
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
print('Pareto plotter ready.')

In [ ]:
# Cell 25 — Plot Gradient Conflict Cosine
def plot_gradient_conflict(root_dir: str):
    grad_df = collect_gradient_logs(root_dir)
    if grad_df.empty: print('No gradient logs found.'); return
    plt.figure(figsize=(7, 4))
    for arm in grad_df['arm'].unique():
        sub = grad_df[grad_df['arm'] == arm]
        grouped = sub.groupby('step')['grad_cosine'].mean().reset_index()
        plt.plot(grouped['step'], grouped['grad_cosine'], label=arm, marker='^')
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.title('Target vs Retained Gradient Alignment Cosine (cos > 0 = Aligned, cos < 0 = Conflict)')
    plt.xlabel('Step'); plt.ylabel('Cosine Similarity'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
print('Gradient conflict plotter ready.')

In [ ]:
# Cell 26 — Generate All Diagnostic Plots
plot_training_curves(cfg.output_dir)
plot_eval_curves(cfg.output_dir)
plot_pareto(cfg.output_dir)
plot_gradient_conflict(cfg.output_dir)